Placement Predictor

In [3]:
# Placement Predictor Challenge
# Random Forest Optimization Comparison + Imbalance Fix

import time
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from scipy.stats import randint
from imblearn.over_sampling import SMOTE

# -------------------------------
# Phase 1: Data Architecture
# -------------------------------

# Generate synthetic dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    weights=[0.9, 0.1],   # 90% placed, 10% at-risk/unplaced
    flip_y=0.01,
    random_state=42
)

# Optional feature names
feature_names = [
    "CGPA", "Internships", "Backlogs", "AptitudeScore", "CommunicationSkills",
    "Projects", "Attendance", "CodingSkills", "Certifications", "Workshops",
    "Hackathons", "TechnicalTest", "MockInterview", "Leadership", "Teamwork",
    "ProblemSolving", "Discipline", "Confidence", "Extracurricular", "Research"
]

X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="AtRisk")   # 1 = at-risk/unplaced, 0 = placed

# Split data: 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Feature scaling
# Fit ONLY on training data to avoid leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Test set shape:", X_test_scaled.shape)
print("Training class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

# -------------------------------
# Phase 2: Baseline Model
# -------------------------------

baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train_scaled, y_train)

y_pred_baseline = baseline_model.predict(X_test_scaled)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)

print("\n--- Baseline Model ---")
print("Accuracy:", round(baseline_accuracy, 4))
print("F1-Score:", round(baseline_f1, 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_baseline))
print(classification_report(y_test, y_pred_baseline, target_names=["Placed", "At-Risk"]))

# -------------------------------
# Phase 3: Balanced Random Forest
# -------------------------------

balanced_model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

balanced_model.fit(X_train_scaled, y_train)

y_pred_balanced = balanced_model.predict(X_test_scaled)

balanced_accuracy = accuracy_score(y_test, y_pred_balanced)
balanced_f1 = f1_score(y_test, y_pred_balanced)

print("\n--- Balanced Random Forest ---")
print("Accuracy:", round(balanced_accuracy, 4))
print("F1-Score:", round(balanced_f1, 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_balanced))
print(classification_report(y_test, y_pred_balanced, target_names=["Placed", "At-Risk"]))

# -------------------------------
# Phase 4: SMOTE on Training Data Only
# -------------------------------

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("\nAfter SMOTE class distribution:")
print(pd.Series(y_train_smote).value_counts())

# -------------------------------
# Phase 5: SMOTE + Balanced Random Forest
# -------------------------------

smote_balanced_model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

smote_balanced_model.fit(X_train_smote, y_train_smote)

y_pred_smote_balanced = smote_balanced_model.predict(X_test_scaled)

smote_balanced_accuracy = accuracy_score(y_test, y_pred_smote_balanced)
smote_balanced_f1 = f1_score(y_test, y_pred_smote_balanced)

print("\n--- SMOTE + Balanced Random Forest ---")
print("Accuracy:", round(smote_balanced_accuracy, 4))
print("F1-Score:", round(smote_balanced_f1, 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_smote_balanced))
print(classification_report(y_test, y_pred_smote_balanced, target_names=["Placed", "At-Risk"]))

# -------------------------------
# Phase 6: Grid Search on SMOTE Data (Optimize F1)
# -------------------------------

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

grid_f1 = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

start_grid = time.time()
grid_f1.fit(X_train_smote, y_train_smote)
end_grid = time.time()

best_grid_model = grid_f1.best_estimator_
y_pred_grid = best_grid_model.predict(X_test_scaled)

grid_accuracy = accuracy_score(y_test, y_pred_grid)
grid_f1_score_test = f1_score(y_test, y_pred_grid)
grid_time = end_grid - start_grid

print("\n--- Grid Search on SMOTE Data (Optimizing F1) ---")
print("Best Parameters:", grid_f1.best_params_)
print("CV Best Score (F1):", round(grid_f1.best_score_, 4))
print("Test Accuracy:", round(grid_accuracy, 4))
print("Test F1-Score:", round(grid_f1_score_test, 4))
print("Time Taken:", round(grid_time, 4), "seconds")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_grid))
print(classification_report(y_test, y_pred_grid, target_names=["Placed", "At-Risk"]))

# -------------------------------
# Phase 7: Randomized Search on SMOTE Data (Optimize F1)
# -------------------------------

param_dist = {
    "n_estimators": randint(10, 501),
    "max_depth": [None, 5, 10, 15, 20, 25, 30],
    "min_samples_split": randint(2, 21),
    "min_samples_leaf": randint(1, 11)
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ),
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1
)

start_rand = time.time()
random_search.fit(X_train_smote, y_train_smote)
end_rand = time.time()

best_random_model = random_search.best_estimator_
y_pred_random = best_random_model.predict(X_test_scaled)

random_accuracy = accuracy_score(y_test, y_pred_random)
random_f1 = f1_score(y_test, y_pred_random)
random_time = end_rand - start_rand

print("\n--- Randomized Search on SMOTE Data (Optimizing F1) ---")
print("Best Parameters:", random_search.best_params_)
print("CV Best Score (F1):", round(random_search.best_score_, 4))
print("Test Accuracy:", round(random_accuracy, 4))
print("Test F1-Score:", round(random_f1, 4))
print("Time Taken:", round(random_time, 4), "seconds")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_random))
print(classification_report(y_test, y_pred_random, target_names=["Placed", "At-Risk"]))

# -------------------------------
# Phase 8: Final Comparison Table
# -------------------------------

comparison_table = pd.DataFrame({
    "Method": [
        "Baseline Random Forest",
        "Balanced Random Forest",
        "SMOTE + Balanced RF",
        "Grid Search (SMOTE + Balanced RF)",
        "Randomized Search (SMOTE + Balanced RF)"
    ],
    "Best Parameters": [
        "Default",
        "class_weight='balanced'",
        "SMOTE + class_weight='balanced'",
        str(grid_f1.best_params_),
        str(random_search.best_params_)
    ],
    "Test Accuracy": [
        round(baseline_accuracy, 4),
        round(balanced_accuracy, 4),
        round(smote_balanced_accuracy, 4),
        round(grid_accuracy, 4),
        round(random_accuracy, 4)
    ],
    "Test F1-Score": [
        round(baseline_f1, 4),
        round(balanced_f1, 4),
        round(smote_balanced_f1, 4),
        round(grid_f1_score_test, 4),
        round(random_f1, 4)
    ],
    "Time Taken (sec)": [
        0,
        0,
        0,
        round(grid_time, 4),
        round(random_time, 4)
    ]
})

print("\n--- Final Comparison Table ---")
print(comparison_table)

# -------------------------------
# Phase 9: Best Final Model
# -------------------------------

best_index = comparison_table["Test F1-Score"].idxmax()
print("\nBest model based on Test F1-Score:")
print(comparison_table.loc[best_index])


Training set shape: (800, 20)
Test set shape: (200, 20)
Training class distribution:
 AtRisk
0    0.8925
1    0.1075
Name: proportion, dtype: float64
Test class distribution:
 AtRisk
0    0.895
1    0.105
Name: proportion, dtype: float64

--- Baseline Model ---
Accuracy: 0.93
F1-Score: 0.5333
Confusion Matrix:
 [[178   1]
 [ 13   8]]
              precision    recall  f1-score   support

      Placed       0.93      0.99      0.96       179
     At-Risk       0.89      0.38      0.53        21

    accuracy                           0.93       200
   macro avg       0.91      0.69      0.75       200
weighted avg       0.93      0.93      0.92       200


--- Balanced Random Forest ---
Accuracy: 0.91
F1-Score: 0.3077
Confusion Matrix:
 [[178   1]
 [ 17   4]]
              precision    recall  f1-score   support

      Placed       0.91      0.99      0.95       179
     At-Risk       0.80      0.19      0.31        21

    accuracy                           0.91       200
   macro avg 